# 01c - auto_LiRPA against Pasado after the fix, both on the same GPU
Protocol: `results/lirpa_postfix_2026-11/PROTOCOL.md`. Run in a fresh runtime (auto_LiRPA is
installed into it). Tiers in priority order; every record is per (tool, network, eps, image)
and resumes after a disconnect.

In [ ]:
# 0.1 Setup: Google Drive, the code at the pinned commit, pinned packages (~3-5 min)
import json, os, subprocess, sys
COMMIT = "UNPINNED-see-next-commit"          # code under test (branch checkpoint)
BRANCH = "checkpoint"
REPO_URL = "https://github.com/dimitrijepesic/Pasado.git"   # default source of the code
DRIVE_ROOT = "/content/drive/MyDrive/pasado_checkpoint_2026_11"
BUNDLE = DRIVE_ROOT + "/checkpoint.bundle"   # optional: used instead of GitHub if present
LOCAL_TEST = os.environ.get("PASADO_NB_LOCAL_TEST", "0") == "1"
PIN_TORCH = ["torch==2.11.0", "torchvision==0.26.0"]        # the stack of the October A100 runs
TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
PINS = ["numpy==2.5.1", "scipy==1.18.0", "scikit-learn==1.9.0", "mpmath==1.4.1", "affapy==0.1"]
if LOCAL_TEST:
    REPO, DRIVE_ROOT = os.environ["PASADO_NB_REPO"], os.environ["PASADO_NB_OUT"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    REPO = "/content/Pasado"
    if not os.path.isdir(REPO + "/.git"):
        src = BUNDLE if os.path.exists(BUNDLE) else REPO_URL
        extra = [] if src == BUNDLE else ["--branch", BRANCH]
        subprocess.run(["git", "clone", *extra, src, REPO], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", COMMIT], check=True)
    import importlib.metadata as md
    if md.version("torch").split("+")[0] != PIN_TORCH[0].split("==")[1]:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIN_TORCH,
                        "--index-url", TORCH_INDEX], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
print("code at", subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                                text=True).stdout.strip())

In [ ]:
# 0.2 Environment record and result folder on Drive (one folder per GPU model)
sys.path.insert(0, os.path.join(REPO, "colab"))
import nbutil
GPU = nbutil.gpu_tag()
OUT = os.path.join(DRIVE_ROOT, "01c_lirpa_postfix", GPU)
CACHE = os.path.join(DRIVE_ROOT, "inductor_cache", GPU)      # compilation is paid once per GPU model
os.makedirs(CACHE, exist_ok=True)
R = nbutil.Runner(REPO, OUT, extra_env={"TORCHINDUCTOR_CACHE_DIR": CACHE})
T = nbutil.LOCAL_TEST
DEV = "cpu" if T else "cuda"
NETS = ["3layer"] if T else ["3layer", "4layer", "5layer", "big"]
if T:
    BIG_CHUNK = 8
else:
    free_mib = int(nbutil.sh("nvidia-smi --query-gpu=memory.free --format=csv,noheader,nounits").split()[0])
    BIG_CHUNK = 64 if free_mib >= 70 * 1024 else 32
print(json.dumps(R.env_record, indent=1))
print("results ->", OUT, "| big chunk", BIG_CHUNK)
if not T:
    print(nbutil.sh("nvidia-smi"))

In [ ]:
# 0.4 auto_LiRPA at a pinned commit (no release tag after v0.3 exists), dependencies without torch
LIRPA_COMMIT = "e3930353a3589f7d10879f1e71223d1eefb57196"
if not T:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    f"git+https://github.com/Verified-Intelligence/auto_LiRPA.git@{LIRPA_COMMIT}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "appdirs", "sortedcontainers", "packaging",
                    "graphviz", "pyyaml"], check=True)
print(subprocess.run([sys.executable, "-c", "import auto_LiRPA, torch; print(auto_LiRPA.__version__, torch.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
# T1-T4 in priority order; each line resumes per (network, eps, image). Planning estimates from the
# August A100 sweep (old per-image times), not results.
TIERS = [
    ("T1_small_10img", ["3layer", "4layer", "5layer"], [0, 2, 4], "0-9", 9, 40),
    ("T2_big_10img_K1", ["big"], [0, 2, 4], "0-9", 1, 8),
    ("T3_small_img10-29", ["3layer", "4layer", "5layer"], [0, 2, 4], "10-29", 9, 80),
    ("T4_all_30img_16eps_K1", ["3layer", "4layer", "5layer", "big"], list(range(16)), "0-29", 1, 110),
]
if T:
    TIERS = [("T1_small_10img", ["3layer"], [2], "0", 3, 1)]
for name, nets, eps, imgs, budget, est in TIERS:
    for net in nets:
        R.run(f"{name}_{net}", ["experiments/lirpa_same_gpu.py", "--device", DEV, "--networks", net,
                                "--eps-indices", *eps, "--images", imgs, "--budget", budget,
                                "--pasado-ks", *([1, 3] if T else [1, 3, 5, 7, 9, 16, 32, 64, 128, 256]),
                                "--big-chunk", BIG_CHUNK, "--out-dir", R.path("records", "x")[:-2]],
              estimate_min=round(est / len(nets), 1))
    subprocess.run([sys.executable, "experiments/analyze_lirpa_postfix.py", R.path("records", "x")[:-2],
                    "--out-md", R.path("tables.md")], cwd=REPO)

In [ ]:
# Optional: auto_LiRPA K=1..9 against the archived August A100 sweep (put bab_sweep_gpu.csv on Drive).
# Meaningful only once all 30 images of an (network, eps) are present (T1 + T3).
import csv, statistics
arch = os.path.join(DRIVE_ROOT, "bab_sweep_gpu.csv")
if os.path.exists(arch):
    recs = [json.loads(l) for l in open(R.path("records", "lirpa.jsonl"))]
    for r in csv.DictReader(open(arch)):
        mine = [x for x in recs if x["network"] == r["network"] and x["eps_index"] == int(r["eps_index"])
                and r["solves"] in x["curve"]]
        if len(mine) == int(r["images_averaged"]):
            new = statistics.mean(x["curve"][r["solves"]]["bound"] for x in mine)
            print(r["network"], r["eps_index"], "K=" + r["solves"], "archived", r["lirpa_bab"], "now", new,
                  "rel diff", abs(new - float(r["lirpa_bab"])) / float(r["lirpa_bab"]))
else:
    print("no archived CSV on Drive - skipped")